In [ ]:
import os
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import datasets, transforms, models
from PIL import Image

import numpy as np
import plotly.graph_objects as go

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
user_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_DIR = Path('data')
TRAIN_DIR = DATA_DIR / 'Train'
TEST_DIR = DATA_DIR / 'Test'
VAL_DIR = DATA_DIR / 'Validation'

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
LEARN_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 20
PATIENCE = 5
VAL_SPLIT = 0.1

In [ ]:
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents = True, exist_ok = True)

assert TRAIN_DIR.exists() and VAL_DIR.exists() and TEST_DIR.exists(), 'Train/Validation/Test folders not found.'
print('Using data at:', DATA_DIR.resolve())

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale = (0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness = 0.1, contrast = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

eval_tf = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean = IMAGENET_MEAN, std = IMAGENET_STD)
])

ds_train = datasets.ImageFolder(TRAIN_DIR, transform = train_tf)
class_names = ds_train.classes
num_classes = len(class_names)

y_full = np.array([y for _, y in ds_train.samples])
strat_split = StratifiedShuffleSplit(n_splits = 1, test_size = VAL_SPLIT, random_state = SEED)
train_idx, val_idx = next(strat_split.split(np.zeros(len(y_full)), y_full))

ds_train = Subset(datasets.ImageFolder(TRAIN_DIR, transform = train_tf), train_idx)
ds_val = Subset(datasets.ImageFolder(TRAIN_DIR, transform = eval_tf), val_idx)
ds_test = datasets.ImageFolder(TEST_DIR, transform = eval_tf)


In [ ]:
print('Classes:', class_names, ' (num_classes =', num_classes, ')')

print(f'Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')

In [ ]:
class_counts = np.bincount(y_full, minlength = num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1.0)

In [ ]:
train_targets = y_full[train_idx]
sample_weights = [class_weights[y] for y in train_targets]

sampler = WeightedRandomSampler(weights = [float(w) for w in sample_weights],
                                num_samples = len(sample_weights), replacement = True)

dl_train = DataLoader(ds_train, batch_size = BATCH_SIZE, sampler = sampler, num_workers = 2, pin_memory = True)
dl_test = DataLoader(ds_test, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)
dl_val = DataLoader(ds_val, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)

In [ ]:
try:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights = weights)
except Exception:
    model = models.resnet18(pretrained = True)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(user_device)

c_weight = torch.tensor(class_weights, dtype = torch.float32)
c_weight = c_weight / c_weight.sum() * c_weight.numel()
c_weight.to(user_device)

criterion = nn.CrossEntropyLoss(weight = c_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr = LEARN_RATE, weight_decay = WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 5)

In [ ]:
def epoch_step(loader, train : bool):
    model.train() if train else model.eval()

    running_loss, running_corrects, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    if len(loader) == 0:
        return 0.0, 0.0, np.array([], dtype = np.int64), np.array([], dtype = np.int64)

    for xb, yb in loader:
        xb, yb = xb.to(user_device), yb.to(user_device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        preds = torch.argmax(logits, dim = 1).view(-1)
        running_loss += loss.item() * xb.size(0)
 
        running_corrects += (preds == yb).sum().item()
        total += xb.size(0)

        preds_flat = torch.argmax(logits, dim = 1).view(-1) 
        yb_flat = yb.view(-1) 

        if preds_flat.numel() == 0:
            continue

        all_preds.extend(preds_flat.detach().cpu().numpy().ravel().tolist())
        all_targets.extend(yb_flat.detach().cpu().numpy().ravel().tolist())
    
    avg_loss = running_loss / max(total, 1)
    avg_acc = running_corrects / max(total, 1)
    
    return avg_loss, avg_acc, np.array(all_preds, dtype = np.int64), np.array(all_targets, dtype = np.int64)

def train_model(epoch = EPOCHS, patience = PATIENCE):
    best_val, best_state, patience_left = float('inf'), None, patience
    hist = {'train_loss' : [],
            'train_acc' : [],
            'val_loss' : [],
            'val_acc' : []}
    
    for e in range(1, epoch + 1):
        tr_loss, tr_acc, _, _ = epoch_step(dl_train, True)
        val_loss, val_acc, _, _ = epoch_step(dl_val, False)

        scheduler.step(val_loss)
        hist['train_loss'].append(tr_loss)
        hist['val_loss'].append(val_loss)
        hist['train_acc'].append(tr_acc)
        hist['val_acc'].append(val_acc)
        print(f'Epoch {e:02d}/{epoch} | train_loss {tr_loss:.4f} val_loss {val_loss:.4f} | train_acc {tr_acc:.3f} val_acc {val_acc:.3f}')
        if val_loss < best_val - 1e-4:
            best_val, best_state, patience_left = val_loss, {k: v.cpu() for k, v in model.state_dict().items()}, patience
            torch.save(best_state, OUT_DIR/'best_model.pt')
            with open(OUT_DIR/'labels.json', 'w') as f:
                json.dump({i: n for i, n in enumerate(class_names)}, f)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('--Early stopping--')
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(user_device)
    
    return hist

In [ ]:
if len(dl_val) == 0:
    print('Warning: validation split is empty')
    PATIENCE = 0

# history = train_model()

In [ ]:
best_model_path = OUT_DIR / 'best_model.pt'
if best_model_path.exists():
    model.load_state_dict(torch.load(best_model_path, map_location = user_device))
    model.to(user_device)
    model.eval()
    print(f"Loaded best model from {best_model_path}")
else:
    print("Best model file not found.")

In [ ]:
# x = list(range(1, len(history['train_loss']) + 1))

In [ ]:
# fig1 = go.Figure()
# fig1.add_scatter(x = x, y = history['train_loss'], mode = 'lines+markers', name = 'train_loss')
# fig1.add_scatter(x = x, y = history['val_loss'], mode = 'lines+markers', name = 'val_loss')
# fig1.update_layout(title = 'Loss', xaxis_title = 'Epoch', yaxis_title = 'Loss')
# fig1.show()

In [ ]:
# fig2 = go.Figure()
# fig2.add_scatter(x = x, y = history['train_acc'], mode = 'lines+markers', name = 'train_acc')
# fig2.add_scatter(x = x, y = history['val_acc'], mode = 'lines+markers', name = 'val_acc')
# fig2.update_layout(title = 'Accuracy', xaxis_title = 'Epoch', yaxis_title = 'Accuracy')
# fig2.show()